In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
import wandb
from datasets import get_dataset
import torch
from utils import save_run, load_run
from funcs import train_epoch, val_epoch, test_epoch
from weight_inits import init_timesformer
from torch.utils.data import DataLoader
from models.vivit_custom.utils.scheduler import WarmupCosineSchedule
from transforms import get_transforms, Div255, SplitEqually

from constants import get_classes
from transformers import TimesformerConfig, TimesformerForVideoClassification
from transformers import ViTForImageClassification
from pytorchvideo.transforms import MixUp
import torch
from torchvision.transforms.v2 import (
    ScaleJitter,
    RandAugment,
    RandomCrop,
    Compose,
    ColorJitter,
    RandomHorizontalFlip,
    Normalize,
    RandomApply,
    UniformTemporalSubsample,
    CenterCrop,
    Lambda,
)
from pytorchvideo.transforms import UniformTemporalSubsample, ApplyTransformToKey

C:\Users\Wojtek\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config = {
    "base_learning_rate": 0.001,
    "label_smoothing": 0.2,
    "epochs": 10,
    "frames": 8,
    "val_epochs": 1,
    "batch_size": 20,
    "weight_decay": 1e-5,
    "load_run": False,
    "init_vit": True,
    "vit_base_size": "tiny",
    "cosine_schedule": True,
    "augmentations": True,
    "advanced_augmentations": True,
    "mixup": True,
    "mixup_alpha": 0.2,
    "dataset": "ucf101",
}

In [ ]:
CLASSES_LIST = get_classes(config["dataset"])

ds_transform_train, ds_transform_valid, test_transform = get_transforms(config, CLASSES_LIST, config['dataset'])

dataset_train, dataset_valid, dataset_test, num_labels = get_dataset(
    config["dataset"], ds_transform_train, ds_transform_valid, test_transform, lazy=False
)

#training_loader = DataLoader(dataset=dataset_train, batch_size=8, shuffle=True)

In [ ]:
transform = Compose(
    transforms=[
        UniformTemporalSubsample(num_samples=config["frames"], temporal_dim=0),
        RandomHorizontalFlip(p=0.5),
        #ScaleJitter(target_size=(240, 320), scale_range=(0.85, 1.33)),
        RandomCrop(size=(224, 224), pad_if_needed=True, fill=128),
        RandomApply(p=1, transforms=[ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.1)]),
        # RandAugment(num_ops=2, magnitude=10, fill=128),
        Div255(),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

In [ ]:
from utils import show_img_chw

In [ ]:
dataset_valid.transform = transform

In [ ]:
show_img_chw(dataset_valid[0][0][0], denormalize=True)

In [8]:
from vit_configs import get_vit_config
from transformers import TimesformerConfig, TimesformerForVideoClassification, VivitConfig, VivitForVideoClassification
from transformers import ViTForImageClassification
from weight_inits import init_timesformer


VIT_CONFIG = get_vit_config('tiny')
vit = ViTForImageClassification.from_pretrained(f"WinKawaks/vit-tiny-patch16-224")
model_config = VivitConfig(num_frames=config["frames"], **VIT_CONFIG, num_labels=51)
vivit = VivitForVideoClassification(model_config)


VIT_CONFIG = get_vit_config('tiny')
timesformer = ViTForImageClassification.from_pretrained(f"WinKawaks/vit-tiny-patch16-224")
model_config = TimesformerConfig(num_frames=config["frames"], **VIT_CONFIG, num_labels=51)
timesformer = TimesformerForVideoClassification(model_config)


C:\Users\Wojtek\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [9]:
timesformer.timesformer.encoder.layer[0].temporal_dense.weight

Parameter containing:
tensor([[ 0.0031, -0.0298,  0.0117,  ..., -0.0140, -0.0277,  0.0027],
        [-0.0047, -0.0097,  0.0117,  ..., -0.0071, -0.0014, -0.0602],
        [-0.0110, -0.0154,  0.0227,  ..., -0.0151,  0.0175,  0.0050],
        ...,
        [ 0.0102,  0.0117,  0.0138,  ..., -0.0317,  0.0098, -0.0180],
        [ 0.0036, -0.0172, -0.0400,  ...,  0.0188, -0.0306,  0.0188],
        [ 0.0141, -0.0031,  0.0055,  ...,  0.0247, -0.0183,  0.0171]],
       requires_grad=True)

In [3]:
from torchvision.datasets import HMDB51, UCF101
from torch.utils.data import Subset
from pytorchvideo.data.hmdb51 import Hmdb51
from pytorchvideo.data.ucf101 import Ucf101
from pytorchvideo.data.clip_sampling import UniformClipSampler
import os
import pickle
from sklearn.model_selection import train_test_split
import numpy as np


CLASSES_LIST = get_classes("ucf101")
ds_transform_train, ds_transform_valid, test_transform = get_transforms(config, CLASSES_LIST, config["dataset"])

In [4]:
import torch
from torchvision.transforms.v2 import (
    ScaleJitter,
    RandAugment,
    RandomCrop,
    Compose,
    ColorJitter,
    RandomHorizontalFlip,
    Normalize,
    RandomApply,
    UniformTemporalSubsample,
    CenterCrop,
    Lambda,
)
from pytorchvideo.transforms import UniformTemporalSubsample, ApplyTransformToKey
from utils import show_img_chw


In [7]:
def swap_dims(x):
    return x.permute(1, 0, 2, 3)

def convert_dtype(x):
    return x.type(torch.uint8)


test_transforms_list = [
    ApplyTransformToKey(
        key="video",
        transform=Compose(
            [
                Lambda(convert_dtype),
                Lambda(swap_dims),
                # SplitEqually(num_frames=64),
                UniformTemporalSubsample(num_samples=8, temporal_dim=0),
                CenterCrop(size=(224, 224)),
                # Div255(),
                # Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ]
        ),
    )
]
test_transform = Compose(test_transforms_list)

dataset = Ucf101(
            data_path=r"datasets/ucf101/test_train_splits/ucfTrainTestlist/testlist01.txt",
            clip_sampler=UniformClipSampler(2),  # sample whole clips
            decode_audio=False,
            video_path_prefix="datasets/ucf101/data/UCF-101",
            transform=test_transform,
        )

In [8]:
cnt = 0
import matplotlib.pyplot as plt

for i in dataset:
    # print(i['video'].shape)
    cnt += 1
    # show_img_chw(i['video'][0])
    # plt.show()
    if cnt > 5:
        break
    print(i['label'])

3
3
3
34
85


In [55]:
loader = DataLoader(
    dataset=dataset,
    batch_size=12,
)

In [57]:
for i in loader:
    print(i['video'].shape)
    print(i['label'])
    break

torch.Size([12, 8, 3, 224, 224])
tensor([89, 51, 51, 51, 51, 51, 35,  3,  3,  3, 77, 77])


In [11]:
dataset

In [12]:
d2 = Subset(dataset, [0, 1])

In [17]:
cnt = 0
import matplotlib.pyplot as plt

for i in d2:
    # print(i['video'].shape)
    cnt += 1
    # show_img_chw(i['video'][0])
    # plt.show()
    if cnt > 100:
        break
    print(i['label'])

NotImplementedError: Subclasses of Dataset should implement __getitem__.